In [1]:
from dataclasses import dataclass, asdict
from typing import Dict

In [2]:
@dataclass
class FinancialInputs:
    # Customer / order assumptions
    monthly_orders: int
    average_order_value: float

    # Margin before delivery and promo cost
    gross_margin_rate: float

    # Promotion
    promo_order_share: float
    average_discount_rate: float

    # Delivery
    delivery_cost_per_order: float

    # Driver / workforce
    number_of_drivers: int
    monthly_driver_cost: float

    # Other monthly fixed costs
    other_fixed_costs: float

    # Available starting capital
    initial_capital: float

In [3]:
def validate_inputs(inputs: FinancialInputs):
    if inputs.monthly_orders < 0:
        raise ValueError(
            "monthly_orders cannot be negative."
        )

    if inputs.average_order_value < 0:
        raise ValueError(
            "average_order_value cannot be negative."
        )

    if not 0 <= inputs.gross_margin_rate <= 1:
        raise ValueError(
            "gross_margin_rate must be between 0 and 1."
        )

    if not 0 <= inputs.promo_order_share <= 1:
        raise ValueError(
            "promo_order_share must be between 0 and 1."
        )

    if not 0 <= inputs.average_discount_rate <= 1:
        raise ValueError(
            "average_discount_rate must be between 0 and 1."
        )

    if inputs.delivery_cost_per_order < 0:
        raise ValueError(
            "delivery_cost_per_order cannot be negative."
        )

    if inputs.number_of_drivers < 0:
        raise ValueError(
            "number_of_drivers cannot be negative."
        )

    if inputs.monthly_driver_cost < 0:
        raise ValueError(
            "monthly_driver_cost cannot be negative."
        )

    if inputs.other_fixed_costs < 0:
        raise ValueError(
            "other_fixed_costs cannot be negative."
        )

    if inputs.initial_capital < 0:
        raise ValueError(
            "initial_capital cannot be negative."
        )

In [9]:
def calculate_financials(
    inputs: FinancialInputs
) -> Dict:

    validate_inputs(inputs)

    # ==========================================
    # 1. Gross transaction value
    # ==========================================

    gross_revenue = (
        inputs.monthly_orders
        * inputs.average_order_value
    )

    # ==========================================
    # 2. Gross profit before promo + delivery
    # ==========================================

    gross_profit_before_delivery = (
        gross_revenue
        * inputs.gross_margin_rate
    )

    # ==========================================
    # 3. Promotional cost
    # ==========================================

    promo_orders = (
        inputs.monthly_orders
        * inputs.promo_order_share
    )

    promo_subsidy = (
        promo_orders
        * inputs.average_order_value
        * inputs.average_discount_rate
    )

    # ==========================================
    # 4. Delivery cost
    # ==========================================

    total_delivery_cost = (
        inputs.monthly_orders
        * inputs.delivery_cost_per_order
    )

    # ==========================================
    # 5. Contribution profit
    # ==========================================

    contribution_profit = (
        gross_profit_before_delivery
        - promo_subsidy
        - total_delivery_cost
    )

    if inputs.monthly_orders > 0:
        contribution_margin_per_order = (
            contribution_profit
            / inputs.monthly_orders
        )
    else:
        contribution_margin_per_order = 0

    # ==========================================
    # 6. Driver cost
    # ==========================================

    total_driver_cost = (
        inputs.number_of_drivers
        * inputs.monthly_driver_cost
    )

    # ==========================================
    # 7. Fixed operating cost
    # ==========================================

    total_fixed_cost = (
        total_driver_cost
        + inputs.other_fixed_costs
    )

    # ==========================================
    # 8. Operating profit / loss
    # ==========================================

    operating_profit = (
        contribution_profit
        - total_fixed_cost
    )

    # ==========================================
    # 9. Monthly cash burn
    # ==========================================

    monthly_burn = max(
        0,
        -operating_profit
    )

    # ==========================================
    # 10. Capital runway
    # ==========================================

    if monthly_burn > 0:
        runway_months = (
            inputs.initial_capital
            / monthly_burn
        )
    else:
        runway_months = None

    # ==========================================
    # 11. Break-even orders
    # ==========================================

    if inputs.monthly_orders > 0:
        unit_contribution = (
            contribution_profit
            / inputs.monthly_orders
        )
    else:
        unit_contribution = 0

    if unit_contribution > 0:
        break_even_orders = (
            total_fixed_cost
            / unit_contribution
        )
    else:
        break_even_orders = None

    # ==========================================
    # 12. Risk flags
    # ==========================================

    risk_flags = []

    if contribution_margin_per_order < 0:
        risk_flags.append(
            "NEGATIVE_UNIT_ECONOMICS"
        )

    elif contribution_margin_per_order == 0:
        risk_flags.append(
            "ZERO_CONTRIBUTION_MARGIN"
        )

    if operating_profit < 0:
        risk_flags.append(
            "MONTHLY_CASH_BURN"
        )

    if (
        runway_months is not None
        and runway_months < 6
    ):
        risk_flags.append(
            "SHORT_CAPITAL_RUNWAY"
        )

    if break_even_orders is None:
        risk_flags.append(
            "NO_BREAK_EVEN_AT_CURRENT_UNIT_ECONOMICS"
        )

    return {
        "inputs": asdict(inputs),

        "results": {
            "gross_revenue":
                round(gross_revenue, 2),

            "gross_profit_before_delivery":
                round(
                    gross_profit_before_delivery,
                    2
                ),

            "promo_subsidy":
                round(promo_subsidy, 2),

            "total_delivery_cost":
                round(total_delivery_cost, 2),

            "contribution_profit":
                round(contribution_profit, 2),

            "contribution_margin_per_order":
                round(
                    contribution_margin_per_order,
                    2
                ),

            "total_driver_cost":
                round(total_driver_cost, 2),

            "total_fixed_cost":
                round(total_fixed_cost, 2),

            "operating_profit":
                round(operating_profit, 2),

            "monthly_burn":
                round(monthly_burn, 2),

            "runway_months":
                (
                    round(runway_months, 2)
                    if runway_months is not None
                    else None
                ),

            "break_even_orders":
                (
                    round(break_even_orders)
                    if break_even_orders is not None
                    else None
                )
        },

        "risk_flags":
            risk_flags
    }

In [10]:
def print_financial_report(
    report: Dict
):
    results = report["results"]

    print("=" * 60)
    print("FINANCIAL STRESS TEST")
    print("=" * 60)

    print(
        "Gross revenue:",
        results["gross_revenue"]
    )

    print(
        "Gross profit before delivery:",
        results[
            "gross_profit_before_delivery"
        ]
    )

    print(
        "Promo subsidy:",
        results["promo_subsidy"]
    )

    print(
        "Delivery cost:",
        results["total_delivery_cost"]
    )

    print(
        "Contribution profit:",
        results["contribution_profit"]
    )

    print(
        "Contribution margin per order:",
        results[
            "contribution_margin_per_order"
        ]
    )

    print(
        "Driver cost:",
        results["total_driver_cost"]
    )

    print(
        "Total fixed cost:",
        results["total_fixed_cost"]
    )

    print(
        "Operating profit:",
        results["operating_profit"]
    )

    print(
        "Monthly burn:",
        results["monthly_burn"]
    )

    runway = results["runway_months"]

    if runway is None:
        print(
            "Capital runway:",
            "Not currently burning capital"
        )
    else:
        print(
            "Capital runway:",
            f"{runway} months"
        )

    break_even = results[
        "break_even_orders"
    ]

    if break_even is None:
        print(
            "Break-even orders:",
            "Not reachable with current "
            "unit economics"
        )
    else:
        print(
            "Break-even orders/month:",
            break_even
        )

    print()
    print("RISK FLAGS:")

    if report["risk_flags"]:
        for risk in report["risk_flags"]:
            print(
                "-",
                risk
            )
    else:
        print("- None")

In [11]:
test_inputs = FinancialInputs(
    monthly_orders=3000,

    average_order_value=150000,

    gross_margin_rate=0.20,

    promo_order_share=0.60,

    average_discount_rate=0.20,

    delivery_cost_per_order=12000,

    number_of_drivers=20,

    monthly_driver_cost=4500000,

    other_fixed_costs=30000000,

    initial_capital=1000000000
)

In [12]:
financial_report = calculate_financials(
    test_inputs
)

financial_report

{'inputs': {'monthly_orders': 3000,
  'average_order_value': 150000,
  'gross_margin_rate': 0.2,
  'promo_order_share': 0.6,
  'average_discount_rate': 0.2,
  'delivery_cost_per_order': 12000,
  'number_of_drivers': 20,
  'monthly_driver_cost': 4500000,
  'other_fixed_costs': 30000000,
  'initial_capital': 1000000000},
 'results': {'gross_revenue': 450000000,
  'gross_profit_before_delivery': 90000000.0,
  'promo_subsidy': 54000000.0,
  'total_delivery_cost': 36000000,
  'contribution_profit': 0.0,
  'contribution_margin_per_order': 0.0,
  'total_driver_cost': 90000000,
  'total_fixed_cost': 120000000,
  'operating_profit': -120000000.0,
  'monthly_burn': 120000000.0,
  'runway_months': 8.33,
  'break_even_orders': None},
 'risk_flags': ['ZERO_CONTRIBUTION_MARGIN',
  'MONTHLY_CASH_BURN',
  'NO_BREAK_EVEN_AT_CURRENT_UNIT_ECONOMICS']}

In [13]:
print_financial_report(
    financial_report
)

FINANCIAL STRESS TEST
Gross revenue: 450000000
Gross profit before delivery: 90000000.0
Promo subsidy: 54000000.0
Delivery cost: 36000000
Contribution profit: 0.0
Contribution margin per order: 0.0
Driver cost: 90000000
Total fixed cost: 120000000
Operating profit: -120000000.0
Monthly burn: 120000000.0
Capital runway: 8.33 months
Break-even orders: Not reachable with current unit economics

RISK FLAGS:
- ZERO_CONTRIBUTION_MARGIN
- MONTHLY_CASH_BURN
- NO_BREAK_EVEN_AT_CURRENT_UNIT_ECONOMICS
